In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor
>>> case Tumor
	DEGs 19572
		Up (#9553)
		Dw (#10019)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    13
1                      IG_C_pseudogene     2
2                            IG_J_gene    11
3                      IG_J_pseudogene     1
4                            IG_V_gene   122
5                      IG_V_pseudogene    42
6                              Mt_tRNA    13
7                                  TEC    96
8                            TR_C_gene     6
9                            TR_D_gene     1
10                           TR_J_gene     9
11                           TR_V_gene    43
12                     TR_V_pseudogene     2
13                              lncRNA  3139
14                               miRNA    77
15                            misc_RNA    15
16              polymorphic_pseudogene     4
17        

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA)

gdc.memory_restriction = False

### Get all programs

In [6]:
force = False
verbose = False

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

"; ".join(prog_list)

'TCGA; MATCH; TARGET; CGCI; CMI; APOLLO; BEATAML1.0; CPTAC; MP2PRT; ALCHEMIST; CCDI; CCG; CDDP_EAGLE; CTSP; EXCEPTIONAL_RESPONDERS; FM; HCMI; MMRF; NCICCR; OHSU; ORGANOID; RC; REBC; TRIO; VAREPOP; WCDT'

In [7]:
PROG_ID = 'ORGANOID'
PROG_ID = 'HCMI'
PROG_ID = 'CPTAC'
PROG_ID = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, force=False, verbose=verbose)
print(len(df_psi))

if PROG_ID == 'TCGA':
    psi_id = 'TCGA-PAAD'
    df2 = df_psi[df_psi.psi_id == psi_id]
else:
    disease_id = 'PAAD'
    df2 = df_psi[df_psi.disease_id == disease_id]

if df2.empty:
    print("Error: No data found for the specified parameters.")
elif len(df2) == 1:
    row = df2.iloc[0]
    psi_id = row.psi_id
    primary_site = row.primary_site

    if PROG_ID == 'TCGA':
        print("Only one entry found for TCGA.")
        disease_id = None
        disease_context = None
        cbioportal_study_id = None
    else:
        disease_id = row.disease_id
        disease_context = row.disease_context
        cbioportal_study_id = row.cbioportal_study_id

    print("\t", psi_id, primary_site)
else:
    print("Multiple entries found for the specified parameters.")

    for _, row in df2.iterrows():
        psi_id = row.psi_id
        disease_id = row.disease_id
        primary_site = row.primary_site
        disease_context = row.disease_context
        cbioportal_study_id = row.cbioportal_study_id
        print("\t", psi_id, disease_id, primary_site, disease_context, cbioportal_study_id)


33
Only one entry found for TCGA.
	 TCGA-PAAD Pancreas


In [8]:
gdc.df_psi.head(3)

,psi_id,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and re...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neoplasms, Transitional Cell Papill...",Bladder Urothelial Carcinoma


### Is there Pancreas?

In [9]:
PSI_ID = 'TCGA-PAAD'
DISEASE_ID = 'PAAD'

verbose=True

if PROG_ID == 'TCGA':
    gdc.set_primary_site(psi_id=PSI_ID, verbose=verbose)
else:
    gdc.set_primary_site(disease_id=DISEASE_ID, verbose=verbose)



-----------------------------
>> psi_id: TCGA-PAAD
>> primary_site: Pancreas
>> disease_id: None
>> disease_type: Ductal and Lobular Neoplasms, Cystic, Mucinous and Serous Neoplasms, Epithelial Neoplasms, NOS, Adenomas and Adenocarcinomas
>> disease_name: Pancreatic Adenocarcinoma

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/mutations
-----------------------------



In [ ]:
force=False
verbose=True

# df_cases, df_all_samples, df_all_mutations = gdc.loop_program_psi_samples(prog_id=PROG_ID, force=force, verbose=verbose)

# print(len(df_cases))
# df_cases.head(6)

### Testing cBioportal web service mutation

In [ ]:
df_list_cases=[]
df_list_samples=[]
barcode_sample_list = []

for ipsi, row in df_psi.iterrows():

    psi_id = row.psi_id
    primary_site = row.primary_site

    gdc.set_primary_site(psi_id)

    print(f"{ipsi}) {psi_id} -{primary_site}", end=" - ")

    df_cases, df_subt, _ = gdc.get_cases_and_subtypes(
        batch_size=200, do_filter=False, force=force, verbose=verbose
    )

    if df_cases.empty:
        print("No cases found for PSI_ID:", psi_id)
        continue

    if isinstance(df_cases, pd.DataFrame):
        df_list_cases.append(df_cases)
    else:
        print("Unexpected type for df_cases:", type(df_cases))
        raise Exception("Stope: unexpected type for df_cases")

    for isubt, row in df_subt.iterrows():
        subtype_global = row.subtype_global
        tumor_class = row.tumor_class
        subtype_tissue = row.subtype_tissue

        df_samples = gdc.get_samples_for_subtypes(
            subtype_global=subtype_global,
            tumor_class=tumor_class,
            subtype_tissue=subtype_tissue,
            batch_size=200,
            force=force,
            verbose=verbose,
        )
        print(f"{isubt}) {gdc.s_case}")

        if df_samples.empty:
            if verbose:
                print(
                    f"No samples found for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}"
                )
            continue

        df_list_samples.append(df_samples)

        df2 = df_samples[
            ~df_samples.sample_type.str.contains("Blood", case=False, na=False)
        ]

        if df2.empty:
            print("No samples having non-blood types for PSI_ID:", psi_id)
            continue

        barcode_sample_list = list(np.unique(df2.barcode_sample))
        if len(barcode_sample_list) > 0:
            break
    if len(barcode_sample_list) > 0:
        break        

In [ ]:
len(barcode_sample_list)

In [ ]:
def resolve_mutation_profile(study_id: str, timeout: int = 60) -> str:
    '''
    here is the cBioPortl endpoint

    there is no PSI_ID 
    one must know the correct study_id

    input:  study_id
    output: molecular_profile_id
    '''

    candidates = [
        f"{study_id}_mutations",
        f"{study_id}_mutations_extended",
    ]

    for mp in candidates:
        print(f"Checking mutation profile: {mp} url: {gdc.url_cbioportal}/molecular-profiles/")
        url = f"{gdc.url_cbioportal}/molecular-profiles/{mp}"
        if requests.get(url, timeout=timeout).ok:
            return mp

    print(f"\n\n------------ No mutation profile found for {study_id} ------------- \n\n")

    return ''

def get_cbio_study_sample_ids(
    study_id: str,
    timeout: int = 60,
) -> list[str]:
    """
    Return all cBioPortal sampleIds for a study.
    """

    url = f"{gdc.url_cbioportal}/studies/{study_id}/samples"

    resp = requests.get(url, timeout=timeout)

    if not resp.ok:
        print(resp.status_code)
        print(resp.text[:2000])
        resp.raise_for_status()

    samples = resp.json()

    return sorted({
        s["sampleId"]
        for s in samples
        if s.get("sampleId")
    })

def resolve_cbio_sample_list(
    study_id: str,
    timeout: int = 60,
) -> str:
    """
    Resolve the best cBioPortal sampleListId for a study.

    Preference:
    1. {study_id}_all
    2. any list ending with _all
    3. any list containing sequenced
    4. largest sample list, if sampleCount is available
    5. first available sample list
    """

    url = f"{gdc.url_cbioportal}/studies/{study_id}/sample-lists"

    resp = requests.get(url, timeout=timeout)

    if not resp.ok:
        print(resp.status_code)
        print(resp.text[:2000])
        resp.raise_for_status()

    sample_lists = resp.json()

    if not sample_lists:
        raise ValueError(f"No sample lists found for study_id={study_id}")

    preferred = f"{study_id}_all"

    for sl in sample_lists:
        if sl.get("sampleListId") == preferred:
            return preferred

    for sl in sample_lists:
        sample_list_id = str(sl.get("sampleListId", ""))
        if sample_list_id.endswith("_all"):
            return sample_list_id

    for sl in sample_lists:
        sample_list_id = str(sl.get("sampleListId", "")).lower()
        name = str(sl.get("name", "")).lower()
        description = str(sl.get("description", "")).lower()

        if (
            "sequenced" in sample_list_id
            or "sequenced" in name
            or "sequenced" in description
        ):
            return sl["sampleListId"]

    sample_lists_with_count = [
        sl for sl in sample_lists
        if sl.get("sampleCount") is not None
    ]

    if sample_lists_with_count:
        sample_lists_with_count = sorted(
            sample_lists_with_count,
            key=lambda x: x.get("sampleCount", 0),
            reverse=True,
        )
        return sample_lists_with_count[0]["sampleListId"]

    return sample_lists[0]["sampleListId"]

def build_cbio_mutation_payload(
    study_id: str,
    molecular_profile_id: str,
    barcode_sample_list: list[str] | None = None,
    timeout: int = 60,
    min_match_fraction: float = 0.50,
) -> dict:
    """
    Build a valid payload for:

        POST /molecular-profiles/{molecular_profile_id}/mutations/fetch

    Strategy:
    - If input sample IDs match cBioPortal sample IDs, use sampleIds.
    - Otherwise, use the best sampleListId.
    """

    barcode_sample_list = barcode_sample_list or []

    input_sample_ids = sorted({
        str(x).strip()
        for x in barcode_sample_list
        if x is not None and str(x).strip()
    })

    if not input_sample_ids:
        sample_list_id = resolve_cbio_sample_list(
            study_id=study_id,
            timeout=timeout,
        )
        return {
            "sampleListId": sample_list_id
        }

    cbio_sample_ids = get_cbio_study_sample_ids(
        study_id=study_id,
        timeout=timeout,
    )

    input_set = set(input_sample_ids)
    cbio_set = set(cbio_sample_ids)

    matched_sample_ids = sorted(input_set & cbio_set)
    missing_sample_ids = sorted(input_set - cbio_set)

    match_fraction = len(matched_sample_ids) / len(input_sample_ids)

    print(f"Study: {study_id}")
    print(f"Molecular profile: {molecular_profile_id}")
    print(f"Input samples: {len(input_sample_ids)}")
    print(f"cBioPortal study samples: {len(cbio_sample_ids)}")
    print(f"Matched samples: {len(matched_sample_ids)}")
    print(f"Missing samples: {len(missing_sample_ids)}")
    print(f"Match fraction: {match_fraction:.3f}")

    if missing_sample_ids:
        print("Example missing sample IDs:")
        print(missing_sample_ids[:10])

    if matched_sample_ids and match_fraction >= min_match_fraction:
        return {
            "sampleIds": matched_sample_ids
        }

    sample_list_id = resolve_cbio_sample_list(
        study_id=study_id,
        timeout=timeout,
    )

    print(
        f"Using sampleListId={sample_list_id} because input sample IDs "
        f"do not sufficiently match cBioPortal sample IDs."
    )

    return {
        "sampleListId": sample_list_id
    }


In [ ]:
study_id = 'paad_cptac_2021'
timeout=60

molecular_profile_id = gdc.resolve_mutation_profile(study_id=study_id)

if molecular_profile_id != '':

    print(">>> molecular_profile_id", molecular_profile_id)

    http = requests.Session()

    url = f"{gdc.url_cbioportal}/molecular-profiles/{molecular_profile_id}/mutations/fetch"

    if gdc.prog_id == 'TCGA':
        payload = {"sampleIds": barcode_sample_list}
    else:
        payload = build_cbio_mutation_payload(
                    study_id=study_id,
                    molecular_profile_id=molecular_profile_id,
                    barcode_sample_list=barcode_sample_list,
                )

    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    resp = http.post(url, json=payload, headers=headers, timeout=timeout)
    resp.raise_for_status()

    # Helpful error message from cBioPortal
    if not resp.ok:
        msg = ""
        try:
            msg = resp.json()
        except Exception:
            msg = resp.text

        raise RuntimeError(
            f"cBioPortal request failed: HTTP {resp.status_code} | "
            f"profile={molecular_profile_id} | details={msg}"
        )

    data = resp.json()
    if not data:
        if verbose:
            print(f"Error: cBioPortal URL: {url}")
            print(
                f"No mutations found for molecular profile '{molecular_profile_id}' barcodes: {barcode_sample_list}."
            )
        df = pd.DataFrame()
    else:
        df = pd.DataFrame(data)
else:
    df = pd.DataFrame()

df

In [ ]:
verbose=False
force=False

imax_samples = 200

df_tumor, df_normal, df_gtex_ctrl = gdc.calc_file_expression_tumor_normal_gtex(
            imax_samples=imax_samples, force=force, verbose=verbose)

print(len(df_tumor), len(df_normal), len(df_gtex_ctrl))
df_tumor.head(3)


In [ ]:
df_normal.head(3)

In [ ]:
df_gtex_ctrl.head(3)